# LLaVA-1.5-7B answers with gaze-foveated patching — your own images and gaze points (no training)

Same as `llava_foveated.ipynb`, but on images you upload and at gaze points you choose: for each image, normal LLaVA (576 image tokens) vs gaze-foveated LLaVA (14 px fovea / 28 px ring / 56 px periphery through APT) at each of your gaze points, on two prompts. Then shows the layouts the model was given.

**Steps:** run the setup cells → upload images → set gaze points → preview (check the red dots are where you want) → choose patch-size levels and `a` values (see the token table) → run LLaVA → view layouts (rows = levels + `a`, columns = gaze points).

Images are resized and centre-cropped to 336×336 (LLaVA's input); gaze points are (u, v) in [0, 1] of that square, u to the right, v down.

**Runtime:** A100 (`Runtime → Change runtime type → A100 GPU`). Everything runs in **fp16**: vision encoder, projector and LLM decoder (no 4-bit); the script prints and checks this. A T4 is too small for LLaVA-7B in fp16 and stops with a message. LLaVA weights (~14 GB) download on first run.

In [ ]:
import os

if not os.path.exists('apt-vlm'):
    !git clone -q https://github.com/shubhamOjha1000/apt-vlm.git
else:
    !git -C apt-vlm pull -q
!git -C apt-vlm log --oneline -1

In [ ]:
# fp16 everywhere -> no bitsandbytes needed. flash-attn is not needed either (eager attention).
!pip install -q -U timm einops ipdb accelerate pillow-heif
!nvidia-smi --query-gpu=name,memory.total --format=csv

import torch
gpu = torch.cuda.get_device_properties(0)
print(f"GPU: {gpu.name}, {gpu.total_memory / 2**30:.0f} GB")
assert gpu.total_memory >= 20 * 2**30, "LLaVA-7B in fp16 needs a >= 20 GB GPU: switch the runtime to A100"

In [ ]:
# Upload images from your computer (jpg, png, webp, bmp, heic). Re-running this cell replaces the previous set.
import shutil
from google.colab import files

UPLOAD_DIR = 'apt-vlm/uploads'
shutil.rmtree(UPLOAD_DIR, ignore_errors=True)
os.makedirs(UPLOAD_DIR)

uploaded = files.upload()
for name, data in uploaded.items():
    with open(os.path.join(UPLOAD_DIR, name), 'wb') as f:
        f.write(data)
print(f'{len(uploaded)} image(s) in {UPLOAD_DIR}:', sorted(os.listdir(UPLOAD_DIR)))

## Set the gaze points
Edit `GAZES` below, run it, then run the preview to check the dots.

In [ ]:
# Gaze points: (u, v) in [0, 1] of the 336x336 crop -- u = left->right, v = top->bottom.
# "*" applies to every image that has no entry of its own. Add an entry per image (file name without
# extension) to give it its own gaze points. Each label becomes one foveated answer + one overlay panel.
GAZES = {
    "*": {
        "centre": (0.5, 0.5),
        "corner": (0.1, 0.1),
        "right": (0.75, 0.4),
    },
    # "my_photo": {"on_face": (0.42, 0.30), "on_sign": (0.80, 0.65)},
}

import json
names = sorted(os.path.splitext(f)[0] for f in os.listdir(UPLOAD_DIR))
unknown = [k for k in GAZES if k != "*" and k not in names]
assert not unknown, f"no uploaded image called {unknown}; image names are {names}"
for name, gazes in GAZES.items():
    for label, (u, v) in gazes.items():
        assert 0 <= u <= 1 and 0 <= v <= 1, f"{name}/{label}: u, v must be in [0, 1]"
with open('apt-vlm/gazes.json', 'w') as f:
    json.dump(GAZES, f, indent=1)
print('image names:', names)
for n in names:
    print(f'  {n}:', GAZES.get(n, GAZES.get("*")))

In [ ]:
# Preview: the 336x336 crop LLaVA sees, a 0.1 grid with u/v labels, and your gaze points.
from PIL import Image as PILImage, ImageDraw
try:
    from pillow_heif import register_heif_opener
    register_heif_opener()
except ImportError:
    pass
from IPython.display import display

def llava_crop(path, size=336):
    """Same as LLaVA's processor: shortest side -> 336 (bicubic), then centre crop."""
    im = PILImage.open(path).convert('RGB')
    s = size / min(im.size)
    im = im.resize((round(im.width * s), round(im.height * s)), PILImage.BICUBIC)
    l, t = (im.width - size) // 2, (im.height - size) // 2
    return im.crop((l, t, l + size, t + size))

files_by_name = {os.path.splitext(f)[0]: os.path.join(UPLOAD_DIR, f) for f in os.listdir(UPLOAD_DIR)}
for n in names:
    im = llava_crop(files_by_name[n]).resize((504, 504), PILImage.BICUBIC)  # 1.5x for readability
    d = ImageDraw.Draw(im, 'RGBA')
    for k in range(1, 10):
        x = k * im.width / 10
        d.line([(x, 0), (x, im.height)], fill=(255, 255, 255, 110))
        d.line([(0, x), (im.width, x)], fill=(255, 255, 255, 110))
        d.text((x + 2, 2), f'{k / 10:.1f}', fill=(255, 255, 0, 255))
        d.text((2, x + 2), f'{k / 10:.1f}', fill=(255, 255, 0, 255))
    for label, (u, v) in GAZES.get(n, GAZES.get("*")).items():
        gx, gy = u * im.width, v * im.height
        d.ellipse([gx - 7, gy - 7, gx + 7, gy + 7], fill=(255, 0, 0, 255), outline=(255, 255, 255, 255))
        d.text((gx + 9, gy - 6), label, fill=(255, 0, 0, 255))
    print(n, '(top labels = u, left labels = v)')
    display(im)

## Choose the patch-size levels and the fall-off values `a`
**Patch-size levels** (`PATCH_LEVELS`): each entry is one setting, starting at 14 px and doubling — `[14]` (no foveation, like normal LLaVA), `[14, 28]`, `[14, 28, 56]`, `[14, 28, 56, 112]`. More levels → coarser periphery → fewer tokens. The image (336 px) must be a multiple of the largest size, so 112 is the maximum.

**`a`** sets where the rings sit (Step 3): small `a` → small fovea, few tokens; large `a` → rings pushed out, more tokens (up to a ceiling). With `[14]` there are no rings, so `a` is ignored.

Every (levels, `a`) pair adds one answer per gaze point per prompt. Token counts change in steps, so different settings can give the *same* layout; the table below shows this before you spend GPU time (identical layouts reuse the same answer).

In [ ]:
PATCH_LEVELS = [[14, 28, 56]]   # e.g. [[14, 28], [14, 28, 56], [14, 28, 56, 112]]
A_VALUES = [10, 40, 160]        # any number of values > 0
FOV_DEG = 110                   # horizontal field of view of the images, degrees (R = FOV / 2)

import sys
sys.path.insert(0, 'apt-vlm')
from src.models.foveation import FoveaConfig, foveated_layout

for lv in PATCH_LEVELS:
    assert lv == [14 * 2 ** i for i in range(len(lv))], f"{lv}: levels must be 14, 28, 56, 112 (start at 14, double each time)"
    assert 336 % lv[-1] == 0, f"{lv}: 336 px is not a multiple of {lv[-1]} (largest allowed is 112)"
assert A_VALUES and all(a > 0 for a in A_VALUES), "a must be > 0"
NUM_SCALES = sorted({len(lv) for lv in PATCH_LEVELS})

def settings():
    for S in NUM_SCALES:
        for a in (A_VALUES if S > 1 else A_VALUES[:1]):
            yield S, a, FoveaConfig(img_size=336, num_scales=S, fov_deg=FOV_DEG, a=a)

def name(cfg):
    sizes = "/".join(map(str, cfg.sizes)) + " px"
    return sizes if cfg.num_scales == 1 else f"{sizes}, a={cfg.a:g}"

print("rings (deg):")
for S, a, cfg in settings():
    print(f"  {name(cfg):<26} " + (", ".join(f"{r:.1f}" for r in cfg.rings_deg) or "none"))

print("\nimage tokens per gaze point (baseline 576), [count per patch size]:")
for n in names:
    print(f"  {n}:")
    for label, uv in GAZES.get(n, GAZES.get("*")).items():
        print(f"    gaze {label}:")
        for S, a, cfg in settings():
            layout = foveated_layout(cfg, [uv])[0]
            c = [sum(1 for *_, s in layout if s == size) for size in cfg.sizes]
            print(f"      {name(cfg):<26} -> {sum(c):>3} [" + "/".join(map(str, c)) + "]")

## Run LLaVA

In [ ]:
# All fp16 (vision encoder, projector, LLM decoder); the first output line reports each part's precision.
# Uses GAZES, PATCH_LEVELS, A_VALUES and FOV_DEG from the cells above. Other option: --max-new-tokens 60
A_ARGS = " ".join(str(a) for a in A_VALUES)
S_ARGS = " ".join(str(S) for S in NUM_SCALES)
!rm -rf apt-vlm/outputs/llava_foveated
!cd apt-vlm && APT_ATTN_IMPL=eager python scripts/llava_foveated.py --image-dir uploads --gazes gazes.json --num-scales {S_ARGS} --a {A_ARGS} --fov {FOV_DEG}

In [ ]:
import glob
from IPython.display import Image, display

for path in sorted(glob.glob('apt-vlm/outputs/llava_foveated/*.png')):
    print(os.path.basename(path), '—', open(path[:-4] + '.gazes.txt').read())
    display(Image(filename=path))